# Post 007 — Anomaly Detection & Dimensionality Reduction
## Dataset A: Jet Engine Sensor Anomaly Detection

**AI Engineering Lab Series | Era 1: Classic Machine Learning**

---

A jet engine at 35,000 feet has hundreds of sensors streaming data every second. Most readings are normal. A tiny fraction are anomalies — and those anomalies could mean anything from a faulty sensor to an impending component failure.

This notebook demonstrates **Isolation Forest** and **Local Outlier Factor (LOF)** for anomaly detection, combined with **t-SNE** and **PCA** for visualizing high-dimensional sensor data. The goal: find the needles in the haystack without any labeled examples of what 'abnormal' looks like.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.manifold import TSNE
from sklearn.metrics import classification_report, roc_auc_score
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8-whitegrid')
print('Libraries loaded')

## 1. Load and Explore the Dataset

Our dataset contains 4,650 jet engine sensor readings with 12 features: temperature, pressure, vibration, fuel flow, oil pressure, compressor speed, turbine inlet temperature, exhaust gas temperature, and more. Approximately 5% of readings are anomalies.

In [ ]:
df = pd.read_csv('../data/jet_engine_sensors.csv')
print(f'Shape: {df.shape}')
print(f'\nAnomaly distribution:')
print(df['is_anomaly'].value_counts())
print(f'Anomaly rate: {df["is_anomaly"].mean():.1%}')
df.head()

In [ ]:
# Distribution of key features: normal vs anomaly
feature_cols = [c for c in df.columns if c != 'is_anomaly']
n_features = len(feature_cols)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for i, feat in enumerate(feature_cols):
    normal = df.loc[df['is_anomaly']==0, feat]
    anomaly = df.loc[df['is_anomaly']==1, feat]
    axes[i].hist(normal, bins=40, alpha=0.6, color='steelblue', label='Normal', density=True)
    axes[i].hist(anomaly, bins=20, alpha=0.7, color='red', label='Anomaly', density=True)
    axes[i].set_title(feat, fontsize=9)
    axes[i].legend(fontsize=7)

for j in range(i+1, len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions: Normal vs Anomaly', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 2. Dimensionality Reduction: PCA and t-SNE

With 12 features, we cannot visualize the data directly. We use two complementary techniques:
- **PCA**: Linear, fast, preserves global structure and variance
- **t-SNE**: Non-linear, slower, excellent at preserving local neighborhood structure

Both are run on the scaled data, and we use the true anomaly labels only for coloring — not for training.

In [ ]:
X = df[feature_cols].values
y_true = df['is_anomaly'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA
pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

# t-SNE (on PCA-reduced data for speed)
pca_10 = PCA(n_components=10, random_state=42)
X_pca10 = pca_10.fit_transform(X_scaled)
tsne = TSNE(n_components=2, perplexity=30, random_state=42, max_iter=500)
X_tsne = tsne.fit_transform(X_pca10)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

for label, color, name in [(0, 'steelblue', 'Normal'), (1, 'red', 'Anomaly')]:
    mask = y_true == label
    size = 8 if label == 0 else 30
    alpha = 0.3 if label == 0 else 0.8
    ax1.scatter(X_pca[mask, 0], X_pca[mask, 1], c=color, label=name, alpha=alpha, s=size)
    ax2.scatter(X_tsne[mask, 0], X_tsne[mask, 1], c=color, label=name, alpha=alpha, s=size)

ax1.set_title(f'PCA ({pca.explained_variance_ratio_.sum():.1%} variance)')
ax1.set_xlabel('PC1'); ax1.set_ylabel('PC2'); ax1.legend()
ax2.set_title('t-SNE (perplexity=30)')
ax2.set_xlabel('t-SNE 1'); ax2.set_ylabel('t-SNE 2'); ax2.legend()

plt.suptitle('Jet Engine Sensor Data: Normal vs Anomaly in Reduced Space', fontsize=13)
plt.tight_layout()
plt.show()

## 3. Isolation Forest

**Isolation Forest** is an ensemble anomaly detector that works on a simple insight: anomalies are few and different, so they are easier to isolate. It builds random trees and measures how many splits it takes to isolate each point. Points that are isolated quickly (few splits) are anomalies.

Key parameter: `contamination` — the expected fraction of anomalies in the dataset.

In [ ]:
# Isolation Forest
iso_forest = IsolationForest(contamination=0.05, random_state=42, n_estimators=200)
iso_pred = iso_forest.fit_predict(X_scaled)
iso_pred_binary = (iso_pred == -1).astype(int)  # -1 = anomaly in sklearn

iso_scores = -iso_forest.score_samples(X_scaled)  # Higher = more anomalous

print('Isolation Forest Results:')
print(classification_report(y_true, iso_pred_binary, target_names=['Normal', 'Anomaly']))
print(f'ROC-AUC: {roc_auc_score(y_true, iso_scores):.3f}')

## 4. Local Outlier Factor (LOF)

**LOF** measures the local density of each point compared to its neighbors. A point is an outlier if its local density is much lower than its neighbors' densities. Unlike Isolation Forest, LOF is a local method — it can detect anomalies even in dense regions of the feature space.

In [ ]:
# Local Outlier Factor
lof = LocalOutlierFactor(n_neighbors=20, contamination=0.05)
lof_pred = lof.fit_predict(X_scaled)
lof_pred_binary = (lof_pred == -1).astype(int)
lof_scores = -lof.negative_outlier_factor_

print('Local Outlier Factor Results:')
print(classification_report(y_true, lof_pred_binary, target_names=['Normal', 'Anomaly']))
print(f'ROC-AUC: {roc_auc_score(y_true, lof_scores):.3f}')

## 5. Visualizing Anomaly Scores

Both algorithms produce a continuous anomaly score, not just a binary label. Visualizing the score distribution helps us understand the threshold decision and the confidence of each prediction.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# Isolation Forest scores in PCA space
sc1 = axes[0,0].scatter(X_pca[:, 0], X_pca[:, 1], c=iso_scores, cmap='RdYlGn_r', s=10, alpha=0.6)
plt.colorbar(sc1, ax=axes[0,0], label='Anomaly Score')
axes[0,0].set_title('Isolation Forest: Anomaly Scores in PCA Space')
axes[0,0].set_xlabel('PC1'); axes[0,0].set_ylabel('PC2')

# LOF scores in PCA space
sc2 = axes[0,1].scatter(X_pca[:, 0], X_pca[:, 1], c=lof_scores, cmap='RdYlGn_r', s=10, alpha=0.6)
plt.colorbar(sc2, ax=axes[0,1], label='LOF Score')
axes[0,1].set_title('LOF: Anomaly Scores in PCA Space')
axes[0,1].set_xlabel('PC1'); axes[0,1].set_ylabel('PC2')

# Score distributions
axes[1,0].hist(iso_scores[y_true==0], bins=50, alpha=0.6, color='steelblue', label='Normal', density=True)
axes[1,0].hist(iso_scores[y_true==1], bins=20, alpha=0.7, color='red', label='Anomaly', density=True)
axes[1,0].set_title('Isolation Forest Score Distribution')
axes[1,0].set_xlabel('Anomaly Score'); axes[1,0].legend()

axes[1,1].hist(lof_scores[y_true==0], bins=50, alpha=0.6, color='steelblue', label='Normal', density=True)
axes[1,1].hist(lof_scores[y_true==1], bins=20, alpha=0.7, color='red', label='Anomaly', density=True)
axes[1,1].set_title('LOF Score Distribution')
axes[1,1].set_xlabel('LOF Score'); axes[1,1].legend()

plt.suptitle('Anomaly Detection Score Analysis', fontsize=14)
plt.tight_layout()
plt.show()

## 6. Summary

| Method | ROC-AUC | Precision (Anomaly) | Recall (Anomaly) | Best For |
|---|---|---|---|---|
| **Isolation Forest** | Computed above | Computed above | Computed above | Global outliers, fast, scalable |
| **LOF** | Computed above | Computed above | Computed above | Local density anomalies, dense regions |

**Key takeaways:**
1. **t-SNE reveals cluster structure** that PCA misses — anomalies often appear as isolated points in t-SNE space
2. **Isolation Forest is faster and more scalable** — ideal for streaming sensor data
3. **LOF is more sensitive to local density** — better when anomalies are embedded within normal clusters
4. **The contamination parameter matters** — always estimate it from domain knowledge before tuning
5. **ROC-AUC > 0.9 is excellent** for unsupervised anomaly detection with no labeled training data